<a href="https://colab.research.google.com/github/jhughes7386/cosc-650-applied-llm-systems/blob/week-07/week-07/week7_adaptation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 7 (starter): The Adaptation Decision and a Production Dataset

No GPU, no fine-tuning. The dataset and the dynamic few-shot selection run locally; the evaluation calls Gemini (guarded: without a key it reports API-BLOCKED, the local parts still run). Cells marked **TODO (you)** are yours. Dependencies: `tiktoken`, `sentence-transformers`. For the eval: `pip install openai`.

## Part 1: Dataset Overview

I used the instructor's synthetic intent-classification dataset with five intents: reminders, weather, music, messaging, and directions. I slightly expanded the music slot values and generated more examples because the original dataset did not produce enough unique examples after deduplication. This allowed the final dataset to meet the required **200 training and 50 evaluation examples**.

In [ ]:
import os, re, random, pathlib
from collections import Counter
random.seed(7)

# TODO (you): replace this generator with your own domain data. Task here: intent classification.
INTENTS={'set_reminder':('remind me to {task} {time}','set a reminder to {task}','ping me to {task} {time}','reminder: {task} {time}'),
  'get_weather':('weather in {city}','will it rain in {city} {time}','how hot in {city}','forecast for {city} {time}'),
  'play_music':('play {x}','put on some {x}','queue up {x}','I want to hear {x}'),
  'send_message':('text {p} that {task}','message {p} about {task}','tell {p} {task}','let {p} know {task}'),
  'get_directions':('directions to {city}','navigate to {city} {time}','route to {city}','take me to {city}')}

SLOT={'task':['call mom','buy milk','submit the report','pay rent','book a flight','walk the dog','renew the license','email the client','restock coffee','back up the laptop'],
  'time':['tomorrow','at 5pm','tonight','this weekend','on monday','at noon','next week','before bed'],
  'city':['Denver','Tokyo','Lagos','Paris','Cairo','Oslo','Lima','Seoul','Madrid','Dublin'],
  'x':['jazz','lofi','Adele','Miles Davis','reggaeton','classical','Fela Kuti','ambient',
      'rock','pop','hip hop','country','blues','R&B','electronic','folk',
      'Taylor Swift','Drake','Beyonce','The Weeknd'],
  'p':['Sam','my brother','the team','Priya','Dad','Marcus','my sister','the group']}

def fill(t): return re.sub(r'\{(\w+)\}', lambda m: random.choice(SLOT[m.group(1)]), t)

raw=[{'text':fill(random.choice(tmpl)),'label':lab} for lab,tmpl in INTENTS.items() for _ in range(300)]
raw+=[{'text':'','label':'set_reminder'}]*4+[{'text':'hi','label':'unknown'}]*3   # noise to clean

print('raw pool:', len(raw))

raw pool: 1507


In [ ]:
VALID=set(INTENTS)
valid=[r for r in raw if r['text'].strip() and r['label'] in VALID]
seen=set(); dedup=[]
for r in valid:
    k=(r['text'].lower(),r['label'])
    if k not in seen: seen.add(k); dedup.append(r)
print(f'dropped invalid {len(raw)-len(valid)}, dropped duplicate {len(valid)-len(dedup)}, clean {len(dedup)}')
by={lab:[r for r in dedup if r['label']==lab] for lab in VALID}
train=[]; eval_=[]
for lab,rows in by.items():
    random.shuffle(rows); train+=rows[:40]; eval_+=rows[40:50]
random.shuffle(train); random.shuffle(eval_)
print('train',len(train),'eval',len(eval_))
print('train label balance:', dict(Counter(r['label'] for r in train)))
import tiktoken
os.environ['TIKTOKEN_CACHE_DIR']=str((pathlib.Path('.')/'.tiktoken_cache').resolve())
enc=tiktoken.get_encoding('cl100k_base'); lens=[len(enc.encode(r['text'])) for r in train]
print(f'token length  min {min(lens)} median {sorted(lens)[len(lens)//2]} max {max(lens)}')
print('format check (text+label present):', all(r['text'] and r['label'] for r in train+eval_))

dropped invalid 7, dropped duplicate 895, clean 605
train 200 eval 50
train label balance: {'set_reminder': 40, 'get_directions': 40, 'get_weather': 40, 'play_music': 40, 'send_message': 40}
token length  min 2 median 5 max 10
format check (text+label present): True


## Part 2: Adapt with dynamic few-shot (local selection)
For each eval input, retrieve the most similar training examples to place in the prompt. This is the no-GPU adaptation, and it reuses Week 5 retrieval.

In [ ]:
os.environ['HF_HOME']=str((pathlib.Path('.')/'.hf_cache').resolve())
from sentence_transformers import SentenceTransformer, util
emb=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
train_emb=emb.encode([r['text'] for r in train], convert_to_tensor=True, normalize_embeddings=True)
def few_shot(query, k=4):
    qe=emb.encode([query], convert_to_tensor=True, normalize_embeddings=True)
    sims=util.cos_sim(qe, train_emb).cpu().numpy()[0]
    return [train[i] for i in sims.argsort()[::-1][:k]]
ex=eval_[0]
print('eval input:', ex['text'], '(true:', ex['label'], ')')
print('retrieved few-shot examples:')
for s in few_shot(ex['text'], 4): print('  ', s['text'], '->', s['label'])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

eval input: I want to hear jazz (true: play_music )
retrieved few-shot examples:
   queue up jazz -> play_music
   I want to hear rock -> play_music
   I want to hear reggaeton -> play_music
   I want to hear blues -> play_music


### Run a zero API key analysis

In [ ]:
from sentence_transformers import util

retrieval_analysis = []

for r in eval_:
    qe = emb.encode(
        [r['text']],
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    sims = util.cos_sim(qe, train_emb).cpu().numpy()[0]
    top_idx = sims.argsort()[::-1][:4]

    retrieved = [train[i] for i in top_idx]
    labels = [x['label'] for x in retrieved]
    similarities = [float(sims[i]) for i in top_idx]

    retrieval_analysis.append({
        'text': r['text'],
        'true_label': r['label'],
        'top_labels': labels,
        'min_similarity': min(similarities),
        'avg_similarity': sum(similarities) / len(similarities)
    })

for r in sorted(retrieval_analysis, key=lambda x: x['min_similarity'])[:15]:
    print(f"\nText: {r['text']}")
    print(f"True: {r['true_label']}")
    print(f"Top labels: {r['top_labels']}")
    print(f"Min similarity: {r['min_similarity']:.3f}")
    print(f"Avg similarity: {r['avg_similarity']:.3f}")


Text: play Fela Kuti
True: play_music
Top labels: ['play_music', 'play_music', 'play_music', 'play_music']
Min similarity: 0.318
Avg similarity: 0.438

Text: play lofi
True: play_music
Top labels: ['play_music', 'play_music', 'play_music', 'play_music']
Min similarity: 0.439
Avg similarity: 0.565

Text: set a reminder to buy milk
True: set_reminder
Top labels: ['set_reminder', 'send_message', 'set_reminder', 'set_reminder']
Min similarity: 0.464
Avg similarity: 0.571

Text: text my brother that buy milk
True: send_message
Top labels: ['send_message', 'set_reminder', 'send_message', 'send_message']
Min similarity: 0.482
Avg similarity: 0.587

Text: I want to hear classical
True: play_music
Top labels: ['play_music', 'play_music', 'play_music', 'play_music']
Min similarity: 0.485
Avg similarity: 0.517

Text: put on some classical
True: play_music
Top labels: ['play_music', 'play_music', 'play_music', 'play_music']
Min similarity: 0.491
Avg similarity: 0.539

Text: text Sam that buy milk

In [ ]:
print("Examples where retrieved labels do not all match the true label:\n")

for r in retrieval_analysis:
    if not all(label == r['true_label'] for label in r['top_labels']):
        print(f"Text: {r['text']}")
        print(f"True: {r['true_label']}")
        print(f"Top labels: {r['top_labels']}")
        print(f"Min similarity: {r['min_similarity']:.3f}")
        print()

Examples where retrieved labels do not all match the true label:

Text: ping me to email the client before bed
True: set_reminder
Top labels: ['set_reminder', 'set_reminder', 'set_reminder', 'send_message']
Min similarity: 0.632

Text: directions to Lima
True: get_directions
Top labels: ['get_directions', 'get_weather', 'get_weather', 'get_directions']
Min similarity: 0.541

Text: text my brother that buy milk
True: send_message
Top labels: ['send_message', 'set_reminder', 'send_message', 'send_message']
Min similarity: 0.482

Text: set a reminder to buy milk
True: set_reminder
Top labels: ['set_reminder', 'send_message', 'set_reminder', 'set_reminder']
Min similarity: 0.464

Text: forecast for Denver tonight
True: get_weather
Top labels: ['get_weather', 'get_weather', 'get_weather', 'get_directions']
Min similarity: 0.684

Text: will it rain in Lagos at noon
True: get_weather
Top labels: ['get_weather', 'get_weather', 'get_directions', 'get_weather']
Min similarity: 0.657

Text: text 

## Part 3: Evaluate, including one failure
This imports my API key, builds the model, and runs test failure call from the previous test results. This is because the free tier has serious limitations

In [16]:
# import api key to make test calls
from google.colab import userdata

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

print('GEMINI_API_KEY exists:', bool(os.environ.get('GEMINI_API_KEY')))

GEMINI_API_KEY exists: True


In [17]:
def gemini_label(prompt, model='gemini-3.5-flash-lite'):
    if not os.environ.get('GEMINI_API_KEY'):
        return None

    from openai import OpenAI

    c = OpenAI(
        api_key=os.environ['GEMINI_API_KEY'],
        base_url='https://generativelanguage.googleapis.com/v1beta/openai/'
    )

    return c.chat.completions.create(
        model=model,
        messages=[{'role': 'user', 'content': prompt}]
    ).choices[0].message.content.strip()


def classify(text, shots=None):
    head = 'Classify the intent into one of: ' + ', '.join(VALID) + '.\n'

    body = ''.join(
        f'Text: {s["text"]}\nIntent: {s["label"]}\n'
        for s in (shots or [])
    )

    return gemini_label(
        head + body + f'Text: {text}\nIntent:'
    )

Cache more results to evaluate

In [27]:
import json
from pathlib import Path

CACHE_PATH = Path("gemini_eval_results.json")

if CACHE_PATH.exists():
    with open(CACHE_PATH, "r") as f:
        eval_results = json.load(f)
else:
    eval_results = {}

# Only evaluate examples that don't already have saved results.
missing = [r for r in eval_ if r["text"] not in eval_results]
batch = missing[:5]

print(f"Cached: {len(eval_results)}")
print(f"New examples to evaluate: {len(batch)}")

for r in batch:
    try:
        zero = classify(r["text"])
        shots = few_shot(r["text"], k=4)
        few = classify(r["text"], shots=shots)

        if zero is not None and few is not None:
            eval_results[r["text"]] = {
                "zero": zero,
                "few": few
            }

            with open(CACHE_PATH, "w") as f:
                json.dump(eval_results, f, indent=2)

            print(f"Saved {len(eval_results)} results")
    except Exception as e:
        print(f"Call failed: {type(e).__name__}: {e}")
        break

print(f"\nTotal cached results: {len(eval_results)}")

Cached: 15
New examples to evaluate: 5
Saved 16 results
Saved 17 results
Saved 18 results
Saved 19 results
Call failed: RateLimitError: Error code: 429 - [{'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 15, model: gemini-3.5-flash-lite\nPlease retry in 34.462955617s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': '

### 3.1 Evaluation metrics

Had to run in batches of 5 because Gemini API would not exceed that and I stopped getting calls because the saved files would not go past 20.

In [28]:
import json
from pathlib import Path

CACHE_PATH = Path('gemini_eval_results.json')

if CACHE_PATH.exists():
    with open(CACHE_PATH, 'r') as f:
        eval_results = json.load(f)
else:
    eval_results = {}

print("Saved evaluation results:", len(eval_results))

Saved evaluation results: 19


View the underperformed slice

In [29]:
def normalize_prediction(pred):
    if not pred:
        return ""

    pred = pred.strip().lower()

    for prefix in ["intent:", "intent is:", "intent is"]:
        if pred.startswith(prefix):
            pred = pred[len(prefix):].strip()
            break

    return pred.strip().rstrip(" .!`")


def is_correct(pred, true_label):
    pred = normalize_prediction(pred)
    true_label = true_label.strip().lower()

    # Accept the label itself, optionally followed by an explanation.
    return (
        pred == true_label
        or pred.startswith(true_label + " ")
        or pred.startswith(true_label + ":")
    )


zero_correct = 0
few_correct = 0
n = 0

for r in eval_:
    result = eval_results.get(r["text"])

    if result is None:
        continue

    true = r["label"]
    zero_correct += is_correct(result["zero"], true)
    few_correct += is_correct(result["few"], true)
    n += 1

print(f"Evaluated: {n}/{len(eval_)}")

if n:
    print(f"Zero-shot accuracy: {zero_correct}/{n} = {zero_correct/n:.1%}")
    print(f"Dynamic few-shot accuracy: {few_correct}/{n} = {few_correct/n:.1%}")
else:
    print("No cached predictions available to score.")

Evaluated: 19/50
Zero-shot accuracy: 19/19 = 100.0%
Dynamic few-shot accuracy: 19/19 = 100.0%


In [30]:
# Compare the intents
from collections import defaultdict

slice_results = defaultdict(lambda: {
    "n": 0,
    "zero_correct": 0,
    "few_correct": 0
})

for r in eval_:
    result = eval_results.get(r["text"])
    if result is None:
        continue

    label = r["label"]
    slice_results[label]["n"] += 1
    slice_results[label]["zero_correct"] += is_correct(
        result["zero"], label
    )
    slice_results[label]["few_correct"] += is_correct(
        result["few"], label
    )

print("Intent-level evaluation slices:\n")

found = False

for label, s in sorted(slice_results.items()):
    if not s["n"]:
        continue

    zero_acc = s["zero_correct"] / s["n"]
    few_acc = s["few_correct"] / s["n"]

    print(
        f"{label}: n={s['n']}, "
        f"zero-shot={zero_acc:.1%}, "
        f"few-shot={few_acc:.1%}"
    )

    if few_acc < zero_acc:
        found = True
        print("  FEW-SHOT UNDERPERFORMED on this slice.")

if not found:
    print("\nNo intent-level underperformance found in these results.")


Intent-level evaluation slices:

get_directions: n=5, zero-shot=100.0%, few-shot=100.0%
get_weather: n=3, zero-shot=100.0%, few-shot=100.0%
play_music: n=6, zero-shot=100.0%, few-shot=100.0%
send_message: n=2, zero-shot=100.0%, few-shot=100.0%
set_reminder: n=3, zero-shot=100.0%, few-shot=100.0%

No intent-level underperformance found in these results.


### 3.2 Failure case

In [ ]:
candidates = [
    "set a reminder to buy milk",
    "text my brother that buy milk",
    "text Sam that buy milk"
]

for text in candidates:
    true_label = next(r['label'] for r in eval_ if r['text'] == text)

    zero_pred = classify(text)

    shots = few_shot(text, 4)
    few_pred = classify(text, shots)

    print("\nText:", text)
    print("True:", true_label)
    print("Zero-shot:", zero_pred)
    print("Few-shot:", few_pred)
    print("Retrieved:")
    for s in shots:
        print(" ", s['text'], "->", s['label'])


Text: set a reminder to buy milk
True: set_reminder
Zero-shot: set_reminder
Few-shot: set_reminder
Retrieved:
  remind me to buy milk on monday -> set_reminder
  text Dad that buy milk -> send_message
  set a reminder to pay rent -> set_reminder
  reminder: restock coffee before bed -> set_reminder

Text: text my brother that buy milk
True: send_message
Zero-shot: Intent: send_message
Few-shot: send_message
Retrieved:
  text Dad that buy milk -> send_message
  remind me to buy milk on monday -> set_reminder
  tell my brother pay rent -> send_message
  text Dad that call mom -> send_message

Text: text Sam that buy milk
True: send_message
Zero-shot: Intent: send_message
Few-shot: send_message
Retrieved:
  text Dad that buy milk -> send_message
  remind me to buy milk on monday -> set_reminder
  message Sam about restock coffee -> send_message
  tell Sam walk the dog -> send_message


In [ ]:
candidates = [
    "directions to Lima",
    "reminder: back up the laptop before bed",
    "ping me to email the client before bed"
]

for text in candidates:
    true_label = next(r['label'] for r in eval_ if r['text'] == text)

    zero_pred = classify(text)

    shots = few_shot(text, 4)
    few_pred = classify(text, shots)

    print("\nText:", text)
    print("True:", true_label)
    print("Zero-shot:", zero_pred)
    print("Few-shot:", few_pred)
    print("Retrieved:")
    for s in shots:
        print(" ", s['text'], "->", s['label'])


Text: directions to Lima
True: get_directions
Zero-shot: Intent: get_directions
Few-shot: get_directions
Retrieved:
  navigate to Lima at noon -> get_directions
  weather in Lima -> get_weather
  will it rain in Lima tomorrow -> get_weather
  directions to Madrid -> get_directions

Text: reminder: back up the laptop before bed
True: set_reminder
Zero-shot: set_reminder
Few-shot: set_reminder
Retrieved:
  remind me to back up the laptop on monday -> set_reminder
  ping me to back up the laptop before bed -> set_reminder
  tell Dad back up the laptop -> send_message
  message Priya about back up the laptop -> send_message

Text: ping me to email the client before bed
True: set_reminder
Zero-shot: Intent: set_reminder
Few-shot: set_reminder
Retrieved:
  ping me to email the client tonight -> set_reminder
  ping me to email the client next week -> set_reminder
  remind me to email the client at noon -> set_reminder
  message the team about email the client -> send_message


In [ ]:
candidates = [
    "play Fela Kuti",
    "play lofi",
    "I want to hear classical"
]

for text in candidates:
    true_label = next(r['label'] for r in eval_ if r['text'] == text)

    zero_pred = classify(text)

    shots = few_shot(text, 4)
    few_pred = classify(text, shots)

    print("\nText:", text)
    print("True:", true_label)
    print("Zero-shot:", zero_pred)
    print("Few-shot:", few_pred)
    print("Retrieved:")
    for s in shots:
        print(" ", s['text'], "->", s['label'])


Text: play Fela Kuti
True: play_music
Zero-shot: Intent: play_music
Few-shot: play_music
Retrieved:
  I want to hear Fela Kuti -> play_music
  play Taylor Swift -> play_music
  play folk -> play_music
  play reggaeton -> play_music

Text: play lofi
True: play_music
Zero-shot: play_music
Few-shot: play_music
Retrieved:
  queue up lofi -> play_music
  put on some lofi -> play_music
  play folk -> play_music
  play R&B -> play_music

Text: I want to hear classical
True: play_music
Zero-shot: play_music
Few-shot: play_music
Retrieved:
  queue up classical -> play_music
  I want to hear reggaeton -> play_music
  I want to hear Adele -> play_music
  I want to hear country -> play_music


In [ ]:
candidates = [
    "weather in Dublin",
    "navigate to Lima tomorrow",
    "tell my brother call mom",
    "let the group know pay rent"
]

for text in candidates:
    true_label = next(r['label'] for r in eval_ if r['text'] == text)

    zero_pred = classify(text)

    shots = few_shot(text, 4)
    few_pred = classify(text, shots)

    print("\nText:", text)
    print("True:", true_label)
    print("Zero-shot:", zero_pred)
    print("Few-shot:", few_pred)
    print("Retrieved:")
    for s in shots:
        print(" ", s['text'], "->", s['label'])


Text: weather in Dublin
True: get_weather
Zero-shot: get_weather
Few-shot: get_weather
Retrieved:
  how hot in Dublin -> get_weather
  forecast for Dublin next week -> get_weather
  navigate to Dublin tomorrow -> get_directions
  navigate to Dublin next week -> get_directions

Text: navigate to Lima tomorrow
True: get_directions
Zero-shot: Intent: get_directions
Few-shot: get_directions
Retrieved:
  navigate to Lima at noon -> get_directions
  will it rain in Lima tomorrow -> get_weather
  weather in Lima -> get_weather
  will it rain in Lima tonight -> get_weather

Text: tell my brother call mom
True: send_message
Zero-shot: Intent: send_message
Few-shot: send_message
Retrieved:
  text Dad that call mom -> send_message
  remind me to call mom this weekend -> set_reminder
  tell my brother pay rent -> send_message
  tell Priya call mom -> send_message

Text: let the group know pay rent
True: send_message
Zero-shot: Intent: send_message
Few-shot: send_message
Retrieved:
  text the grou

In [ ]:
from collections import Counter

risk_analysis = []

for r in eval_:
    shots = few_shot(r['text'], 4)

    labels = [s['label'] for s in shots]
    label_counts = Counter(labels)

    true_count = label_counts[r['label']]
    wrong_count = 4 - true_count

    # Higher risk = more wrong-label retrieved examples
    # and weaker minimum similarity
    qe = emb.encode(
        [r['text']],
        convert_to_tensor=True,
        normalize_embeddings=True
    )

    sims = util.cos_sim(qe, train_emb).cpu().numpy()[0]

    top_idx = sims.argsort()[::-1][:4]
    top_sims = [float(sims[i]) for i in top_idx]

    risk_analysis.append({
        'text': r['text'],
        'true_label': r['label'],
        'labels': labels,
        'true_count': true_count,
        'wrong_count': wrong_count,
        'min_similarity': min(top_sims),
        'avg_similarity': sum(top_sims) / 4
    })

# Sort by:
# 1. most wrong-label examples
# 2. lowest minimum similarity
risk_analysis.sort(
    key=lambda x: (-x['wrong_count'], x['min_similarity'])
)

print("Highest-risk retrieval cases:\n")

for r in risk_analysis:
    if r['wrong_count'] > 0:
        print(f"Text: {r['text']}")
        print(f"True: {r['true_label']}")
        print(f"Retrieved labels: {r['labels']}")
        print(f"Wrong-label examples: {r['wrong_count']}/4")
        print(f"Min similarity: {r['min_similarity']:.3f}")
        print(f"Avg similarity: {r['avg_similarity']:.3f}")
        print()

Highest-risk retrieval cases:

Text: navigate to Lima tomorrow
True: get_directions
Retrieved labels: ['get_directions', 'get_weather', 'get_weather', 'get_weather']
Wrong-label examples: 3/4
Min similarity: 0.658
Avg similarity: 0.727

Text: directions to Lima
True: get_directions
Retrieved labels: ['get_directions', 'get_weather', 'get_weather', 'get_directions']
Wrong-label examples: 2/4
Min similarity: 0.541
Avg similarity: 0.627

Text: reminder: back up the laptop before bed
True: set_reminder
Retrieved labels: ['set_reminder', 'set_reminder', 'send_message', 'send_message']
Wrong-label examples: 2/4
Min similarity: 0.549
Avg similarity: 0.652

Text: remind me to back up the laptop tonight
True: set_reminder
Retrieved labels: ['set_reminder', 'set_reminder', 'send_message', 'send_message']
Wrong-label examples: 2/4
Min similarity: 0.605
Avg similarity: 0.696

Text: text the group that call mom
True: send_message
Retrieved labels: ['send_message', 'send_message', 'set_reminder', 's

In [ ]:
text = "remind me to submit the report tomorrow"

true_label = next(r['label'] for r in eval_ if r['text'] == text)

zero_pred = classify(text)

shots = few_shot(text, 4)
few_pred = classify(text, shots)

print("Text:", text)
print("True:", true_label)
print("Zero-shot:", zero_pred)
print("Few-shot:", few_pred)

print("Retrieved:")
for s in shots:
    print(" ", s['text'], "->", s['label'])

Text: remind me to submit the report tomorrow
True: set_reminder
Zero-shot: set_reminder
Few-shot: set_reminder
Retrieved:
  remind me to submit the report tonight -> set_reminder
  remind me to submit the report on monday -> set_reminder
  let my brother know submit the report -> send_message
  tell my sister submit the report -> send_message


### Part 3: Evaluation

I compared zero-shot classification with dynamic few-shot classification using accuracy as the evaluation metric. Due to Gemini API limitations, I was only able to complete 19 of the 50 evaluation examples before the API stopped accepting additional calls.

- Zero-shot accuracy: **19/19 = 100.0%**
- Dynamic few-shot accuracy: **19/19 = 100.0%**

Both methods achieved the same accuracy on the completed evaluation examples. The original zero-shot accuracy was incorrectly scored because some correct answers started with `Intent:`. After correcting the scoring logic, those answers were counted correctly. These results should be treated as a partial evaluation rather than the full 50-example evaluation.

One potential failure mode came from retrieval. For example, `navigate to Lima tomorrow` had the correct label of `get_directions`, but three of the four retrieved examples were labeled `get_weather`. This shows that retrieval can provide misleading information when different intents have similar wording. In this case, however, the model still predicted the correct intent, so this was a retrieval-risk case rather than an observed few-shot failure.

Another example was `remind me to submit the report tomorrow`. The correct intent was `set_reminder`, but the retrieved examples included both `set_reminder` and `send_message` labels. This is a case where the retrieval step introduced ambiguity because phrases such as “submit the report” can appear in both reminder and messaging examples. The model still predicted the correct intent, but the mixed retrieval shows how dynamic few-shot performance depends on the quality of the examples selected for the prompt.

I also analyzed accuracy by intent. All five intent categories achieved 100% accuracy for both methods on the evaluated examples. I did not observe an intent-level slice where dynamic few-shot performed worse than zero-shot. Although the retrieval analysis identified potential risks, the available results do not demonstrate an actual accuracy failure.

The original Gemini 2.5 Flash Lite model from the demo notebook was unavailable to my API account, so I used Gemini 3.5 Flash Lite for the evaluation. It is possible that the model contributed to the strong results, but I cannot determine that from this evaluation.

## Part 4: Decision Memo

Based on my results, I would consider dynamic few-shot prompting for this task, but I cannot say it performed better than zero-shot. Both methods reached **100.0% accuracy** on the 19 examples I was able to evaluate. Since I was unable to complete all 50 examples due to Gemini API limitations, I would treat these results as preliminary rather than enough evidence to make a final decision.

I would not choose to do nothing without further testing. Both methods performed well on the evaluated examples, but I did see some cases where the retrieved examples had the wrong intent. For example, some reminder examples retrieved messaging examples, and some directions examples retrieved weather examples. If this happened more often with a larger or less consistent dataset, it could become a bigger problem for the few-shot approach.

For now, dynamic few-shot is a reasonable option to continue testing because it does not require a GPU or fine-tuning, and retrieval runs locally. However, my results do not show an accuracy improvement over zero-shot. My main concern would be improving the quality and variety of the training data so the retrieved examples are more reliable.

I would only consider fine-tuning if I had a much larger and more realistic dataset and further evaluation showed that prompting and retrieval were not meeting the task's needs. With the results I have now, fine-tuning would not be worth the extra cost and complexity.

## Part 4 and 5: decision memo, submit
**TODO (you):** write a short memo weighing prompt, RAG, dynamic few-shot, fine-tuning, and doing nothing for your task, grounded in your numbers. Commit the dataset, update your agent context file, open a pull request with a research note. Rubric: dataset + quality report (30), dynamic few-shot (25), before/after + failure slice (20), decision memo (15), PR hygiene (10).